<a href="https://colab.research.google.com/github/PalakPrajapati346/WORKSHOP-2/blob/main/flipkartgridlockelite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

In [17]:
try:
    df_train_input = pd.read_csv('train.csv')
    df_test_input = pd.read_csv('test.csv')
    print("Files loaded successfully.")
except FileNotFoundError:
    print("Error: train.csv or test.csv not found in the current directory!")

Files loaded successfully.


In [20]:
def robust_pipeline(df_raw, is_training=True, target_mapping=None):
    df_p = df_raw.copy()
    df_p['timestamp'] = pd.to_datetime(df_p['timestamp'], format='%H:%M')
    df_p['hour'] = df_p['timestamp'].dt.hour

    # Target Encoding for Geohash
    if is_training:
        target_mapping = df_train_input.groupby('geohash')['demand'].mean().to_dict()

    df_p['geo_signal'] = df_p['geohash'].map(target_mapping).fillna(df_train_input['demand'].mean())

    # Categorical Encoding
    encoder = LabelEncoder()
    for col in ['RoadType', 'Weather', 'LargeVehicles', 'Landmarks']:
        df_p[col] = encoder.fit_transform(df_p[col].astype(str))

    # Drop identifying/raw columns
    drops = ['timestamp', 'geohash', 'Index', 'demand']
    return df_p.drop([c for c in drops if c in df_p.columns], axis=1), target_mapping

# 3. PREPARE FEATUR

In [21]:
X_full_train, signal_map = robust_pipeline(df_train_input, is_training=True)
y_full_train = df_train_input['demand']
X_full_test, _ = robust_pipeline(df_test_input, is_training=False, target_mapping=signal_map)

# Ensure columns match exactly
X_full_test = X_full_test[X_full_train.columns]

In [22]:
lgb_params = {
    'boosting_type': 'gbdt',
    'objective': 'regression',
    'metric': 'rmse',
    'learning_rate': 0.005,    # Very slow learning for 92.88%+ target
    'num_leaves': 1023,        # High complexity to capture nuances
    'feature_fraction': 0.9,
    'bagging_fraction': 0.9,
    'bagging_freq': 5,
    'n_estimators': 10000,     # High estimator count
    'min_data_in_leaf': 10,
    'lambda_l2': 1.0,
    'verbosity': -1,
    'random_state': 42
}

In [23]:
kf_obj = KFold(n_splits=10, shuffle=True, random_state=42)
oof_val_predictions = np.zeros(len(X_full_train))
final_test_predictions = np.zeros(len(X_full_test))

In [24]:
print("Starting 10-Fold training for maximum stability...")

for fold_idx, (t_idx, v_idx) in enumerate(kf_obj.split(X_full_train, y_full_train)):
    xt, xv = X_full_train.iloc[t_idx], X_full_train.iloc[v_idx]
    yt, yv = y_full_train.iloc[t_idx], y_full_train.iloc[v_idx]

    fold_model = lgb.LGBMRegressor(**lgb_params)
    fold_model.fit(xt, yt,
                   eval_set=[(xv, yv)],
                   callbacks=[lgb.early_stopping(stopping_rounds=200),
                              lgb.log_evaluation(period=500)])

    oof_val_predictions[v_idx] = fold_model.predict(xv)
    final_test_predictions += fold_model.predict(X_full_test) / 10
    print(f"Fold {fold_idx + 1} complete.")

Starting 10-Fold training for maximum stability...
Training until validation scores don't improve for 200 rounds
[500]	valid_0's rmse: 0.0389327
[1000]	valid_0's rmse: 0.0366614
Early stopping, best iteration is:
[950]	valid_0's rmse: 0.0366414
Fold 1 complete.
Training until validation scores don't improve for 200 rounds
[500]	valid_0's rmse: 0.0392478
[1000]	valid_0's rmse: 0.036722
Early stopping, best iteration is:
[935]	valid_0's rmse: 0.0366748
Fold 2 complete.
Training until validation scores don't improve for 200 rounds
[500]	valid_0's rmse: 0.0406389
[1000]	valid_0's rmse: 0.0388197
Early stopping, best iteration is:
[827]	valid_0's rmse: 0.0387209
Fold 3 complete.
Training until validation scores don't improve for 200 rounds
[500]	valid_0's rmse: 0.0400039
[1000]	valid_0's rmse: 0.0374177
Early stopping, best iteration is:
[1002]	valid_0's rmse: 0.0374173
Fold 4 complete.
Training until validation scores don't improve for 200 rounds
[500]	valid_0's rmse: 0.0378962
[1000]	vali

In [25]:
final_r2_val = r2_score(y_full_train, oof_val_predictions)
print(f"\n--- VALIDATION RESULTS ---")
print(f"Local Projected Score: {max(0, 100 * final_r2_val):.4f}")


--- VALIDATION RESULTS ---
Local Projected Score: 93.2880


In [27]:
submission_df = pd.DataFrame({
    'Index': df_test_input['Index'],
    'demand': final_test_predictions
})
submission_df.to_csv('submission.csv', index=False)
print("\n'submission.csv' is ready for upload!")


'submission.csv' is ready for upload!


In [ ]:

!pip freeze > requirements.txt